# Hindi TTS Benchmark
### Comprehensive evaluation of Hindi Text-to-Speech models

**Dataset:** `SPRINGLab/IndicTTS-Hindi` — 11,825 high-quality Hindi utterances (IIT Madras),
both male and female speakers, 22 kHz. Ground-truth audio enables reference-based metrics.

**Models benchmarked (all public, zero gating, standard pip installs):**

| # | Model | Description | Install |
|---|---|---|---|
| 1 | `facebook/mms-tts-hin` | Meta MMS-VITS Hindi | `transformers` only |
| 2 | `ai4bharat/indic-parler-tts` | Indic Parler-TTS (HF+AI4Bharat) | `parler-tts` |
| 3 | `hexgrad/Kokoro-82M` | Kokoro with Hindi voice | `kokoro` + `misaki[hi]` |

**Metrics:**

| Category | Metric | Direction |
|---|---|---|
| Intelligibility | CER — Character Error Rate (Whisper large-v3, Hindi) | ↓ lower |
| Intelligibility | WER — Word Error Rate (Whisper large-v3, Hindi) | ↓ lower |
| Spectral | MCD — Mel Cepstral Distortion (DTW-aligned) | ↓ lower |
| Perceptual | UTMOS — Neural MOS predictor (1–5) | ↑ higher |
| Signal | PESQ — Perceptual speech quality | ↑ higher |
| Signal | STOI — Short-time objective intelligibility | ↑ higher |
| Prosody | F0 RMSE — Pitch error in cents | ↓ lower |
| Efficiency | RTF — Real-Time Factor | ↓ lower |

> **Kaggle setup:** Enable Internet and set Accelerator to GPU (T4) for best performance.
> MMS-VITS and Kokoro run well on CPU; Indic Parler-TTS benefits from GPU.


## 1. Install Dependencies

In [ ]:
# espeak-ng must be installed before any TTS import that needs it
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
!espeak-ng --version

# Core scientific stack
%pip install -q --no-warn-conflicts \
    transformers datasets accelerate soundfile \
    librosa scipy numpy matplotlib pandas tqdm

# Indic Parler-TTS  (HuggingFace x AI4Bharat, Apache 2.0)
%pip install -q 'git+https://github.com/huggingface/parler-tts.git'

# Kokoro TTS (Apache 2.0) with Hindi phonemiser
%pip install -q 'kokoro>=0.9.4' 'misaki[hi]'

# openai-whisper for Hindi ASR (WER/CER) — install with --no-deps
# to avoid downgrading transformers
%pip install -q --no-deps openai-whisper
%pip install -q --no-warn-conflicts tiktoken numba more-itertools

# Metrics
%pip install -q jiwer utmos pesq pystoi dtw-python

## 2. Imports & Configuration

In [ ]:
import os, time, warnings, json
import numpy as np
import pandas as pd
import torch
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
REF_SR      = 22050   # IndicTTS-Hindi native sample rate
N_SAMPLES   = 50      # utterances per model; raise to 200+ for publication-quality results
SEED        = 42
OUT_DIR     = Path('hindi_tts_outputs')
OUT_DIR.mkdir(exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device : {DEVICE}')
print(f'Samples: {N_SAMPLES} per model')
print(f'Output : {OUT_DIR}/')

## 3. Dataset — SPRINGLab/IndicTTS-Hindi

The IndicTTS-Hindi dataset is derived from the Indic TTS Database developed by the
Speech Technology Consortium at IIT Madras. It contains high-quality Hindi speech recordings
from both male and female speakers with corresponding text transcriptions, with 11,825 utterances.

The dataset has three columns: `audio` (wav array + sample rate), `text` (Devanagari),
and `gender` (female=0, male=1). We sample equally from both genders for balanced evaluation.


In [ ]:
from datasets import load_dataset

print('Loading SPRINGLab/IndicTTS-Hindi ...')
ds = load_dataset('SPRINGLab/IndicTTS-Hindi', split='train', trust_remote_code=True)
print(f'Full dataset: {len(ds)} utterances | columns: {ds.column_names}')

# Stratified sample: equal male/female
half = N_SAMPLES // 2
female = ds.filter(lambda x: x['gender'] == 0).shuffle(seed=SEED).select(range(half))
male   = ds.filter(lambda x: x['gender'] == 1).shuffle(seed=SEED).select(range(half))
from datasets import concatenate_datasets
bench = concatenate_datasets([female, male]).shuffle(seed=SEED)

# Extract texts and reference audio
texts      = [item['text'] for item in bench]
genders    = ['F' if item['gender'] == 0 else 'M' for item in bench]
ref_audios = []

ref_dir = OUT_DIR / 'reference'
ref_dir.mkdir(exist_ok=True)

for i, item in enumerate(bench):
    wav = np.array(item['audio']['array'], dtype=np.float32)
    sr  = item['audio']['sampling_rate']
    if sr != REF_SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=REF_SR)
    ref_audios.append(wav)
    sf.write(ref_dir / f'{i:04d}.wav', wav, REF_SR)

print(f'Benchmark set: {len(texts)} utterances ({half}F + {half}M)')
print(f'Example text : {texts[0]}')
print(f'Reference dur: {np.mean([len(w)/REF_SR for w in ref_audios]):.1f}s avg')

In [ ]:
# Quick EDA on the benchmark subset
durations = [len(w) / REF_SR for w in ref_audios]
char_lens = [len(t) for t in texts]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('IndicTTS-Hindi — Benchmark Subset EDA', fontsize=13, fontweight='bold')

axes[0].hist(durations, bins=20, color='#4C72B0', edgecolor='white')
axes[0].set_title('Audio Duration Distribution'); axes[0].set_xlabel('Duration (s)')
axes[0].axvline(np.mean(durations), color='red', linestyle='--', label=f'mean={np.mean(durations):.1f}s')
axes[0].legend()

axes[1].hist(char_lens, bins=20, color='#DD8452', edgecolor='white')
axes[1].set_title('Text Length Distribution'); axes[1].set_xlabel('Characters')
axes[1].axvline(np.mean(char_lens), color='red', linestyle='--', label=f'mean={np.mean(char_lens):.0f}')
axes[1].legend()

gender_counts = {'Female': genders.count('F'), 'Male': genders.count('M')}
axes[2].bar(gender_counts.keys(), gender_counts.values(), color=['#C44E52','#55A868'], edgecolor='white')
axes[2].set_title('Gender Distribution')

for ax in axes: ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('hindi_dataset_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Duration: min={min(durations):.1f}s  max={max(durations):.1f}s  mean={np.mean(durations):.1f}s')

## 4. Metric Utilities

### Hindi-specific notes
- **CER** is used alongside WER because Hindi is written in Devanagari — character-level errors
  are more informative than word-level for morphologically complex agglutinative text.
- **Whisper large-v3** has strong Hindi ASR support (ISO code `hi`) and handles Devanagari natively.
- **MCD** uses DTW alignment from `dtw-python` (C-backed), not `fastdtw` (too slow).
- **UTMOS** is reference-free — useful here because MMS-VITS and Parler-TTS produce very
  different speakers from the reference, making reference-based metrics partly unfair.


In [ ]:
import whisper as _w_lib
from jiwer import wer as _wer_fn, cer as _cer_fn
from pesq import pesq as _pesq_fn
from pystoi import stoi as _stoi_fn
from dtw import dtw as _dtw_fn
import utmos

print('Loading Whisper large-v3 (Hindi ASR) ...')
_whisper = _w_lib.load_model('large-v3', device=DEVICE)
print('Whisper loaded.')

print('Loading UTMOS ...')
_utmos_model = utmos.Score()    # no device arg — auto-detects
print('UTMOS loaded.')


# ─── ASR transcription ────────────────────────────────────────────────────────
def _transcribe_hindi(wav: np.ndarray, sr: int) -> str:
    tmp = '/tmp/_hindi_eval.wav'
    if sr != 16000:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=16000)
    sf.write(tmp, wav.astype(np.float32), 16000)
    result = _whisper.transcribe(tmp, language='hi', fp16=(DEVICE == 'cuda'))
    return result['text'].strip()


# ─── MCD ──────────────────────────────────────────────────────────────────────
def _mcd(synth: np.ndarray, ref: np.ndarray, sr: int, n_mfcc: int = 25) -> float:
    def mfcc(w):
        return librosa.feature.mfcc(y=w, sr=sr, n_mfcc=n_mfcc,
                                     n_fft=1024, hop_length=256).T
    cs, cr = mfcc(synth), mfcc(ref)
    aln = _dtw_fn(cs, cr, keep_internals=True)
    diff = cs[aln.index1] - cr[aln.index2]
    return float((10.0 / np.log(10.0)) * np.sqrt(2.0 * np.sum(diff**2, axis=1)).mean())


# ─── F0 RMSE ──────────────────────────────────────────────────────────────────
def _f0_rmse(synth: np.ndarray, ref: np.ndarray, sr: int) -> float:
    def get_f0(w):
        f0, _, _ = librosa.pyin(w, fmin=60, fmax=500, sr=sr, hop_length=256)
        return f0[~np.isnan(f0)]
    fs, fr = get_f0(synth), get_f0(ref)
    if len(fs) == 0 or len(fr) == 0:
        return float('nan')
    n = min(len(fs), len(fr))
    fs = np.interp(np.linspace(0,1,n), np.linspace(0,1,len(fs)), fs)
    fr = np.interp(np.linspace(0,1,n), np.linspace(0,1,len(fr)), fr)
    with np.errstate(divide='ignore', invalid='ignore'):
        cents = 1200.0 * np.log2(np.where(fr > 0, fs/fr, np.nan))
    valid = np.isfinite(cents)
    return float(np.sqrt(np.mean(cents[valid]**2))) if valid.sum() > 0 else float('nan')


# ─── UTMOS ────────────────────────────────────────────────────────────────────
def _utmos(wav: np.ndarray, sr: int) -> float:
    try:
        w16 = librosa.resample(wav, orig_sr=sr, target_sr=16000) if sr != 16000 else wav
        return float(_utmos_model.calculate_wav(w16.astype(np.float32), sample_rate=16000))
    except Exception:
        return float('nan')


# ─── PESQ ─────────────────────────────────────────────────────────────────────
def _pesq(synth: np.ndarray, ref: np.ndarray, sr: int) -> float:
    try:
        if sr != 16000:
            synth = librosa.resample(synth, orig_sr=sr, target_sr=16000)
            ref   = librosa.resample(ref,   orig_sr=sr, target_sr=16000)
        synth = synth.astype(np.float32);  ref = ref.astype(np.float32)
        for a in [synth, ref]:
            mx = np.abs(a).max()
            if mx > 0: a /= mx
        n = min(len(synth), len(ref))
        return float(_pesq_fn(16000, ref[:n], synth[:n], 'wb'))
    except Exception:
        return float('nan')


# ─── STOI ─────────────────────────────────────────────────────────────────────
def _stoi(synth: np.ndarray, ref: np.ndarray, sr: int) -> float:
    try:
        if sr != 16000:
            synth = librosa.resample(synth, orig_sr=sr, target_sr=16000)
            ref   = librosa.resample(ref,   orig_sr=sr, target_sr=16000)
        n = min(len(synth), len(ref))
        return float(_stoi_fn(ref[:n], synth[:n], 16000, extended=False))
    except Exception:
        return float('nan')


# ─── Speaking rate ─────────────────────────────────────────────────────────────
def _rate(wav: np.ndarray, sr: int) -> float:
    onsets = librosa.onset.onset_detect(y=wav, sr=sr, units='time')
    dur = len(wav) / sr
    return len(onsets) / dur if dur > 0 else 0.0


# ─── Master evaluator ──────────────────────────────────────────────────────────
def evaluate_model(name, synth_wavs, synth_sr, texts, ref_wavs, synth_times):
    print(f'\nEvaluating {name} ...')
    wers, cers, mcds, mos, pesqs, stois, f0s, rates, rtfs = \
        [], [], [], [], [], [], [], [], []

    for synth, ref, text, t in tqdm(
            zip(synth_wavs, ref_wavs, texts, synth_times),
            total=len(texts), desc=name, leave=False):

        # Resample synth to reference SR for fair comparison
        s = librosa.resample(synth, orig_sr=synth_sr, target_sr=REF_SR) \
            if synth_sr != REF_SR else synth.copy()

        # WER / CER  (Whisper Hindi)
        hyp = _transcribe_hindi(s, REF_SR)
        ref_clean = text.strip()
        try:
            wers.append(min(_wer_fn(ref_clean, hyp), 1.0))
            cers.append(min(_cer_fn(ref_clean, hyp), 1.0))
        except Exception:
            wers.append(1.0); cers.append(1.0)

        # MCD
        try: mcds.append(_mcd(s, ref, REF_SR))
        except Exception: mcds.append(float('nan'))

        mos.append(_utmos(s, REF_SR))
        pesqs.append(_pesq(s, ref, REF_SR))
        stois.append(_stoi(s, ref, REF_SR))

        try: f0s.append(_f0_rmse(s, ref, REF_SR))
        except Exception: f0s.append(float('nan'))

        rates.append(_rate(s, REF_SR))

        dur = len(s) / REF_SR
        rtfs.append(t / dur if dur > 0 else float('nan'))

    def sm(lst):
        v = [x for x in lst if np.isfinite(x)]
        return round(float(np.mean(v)), 4) if v else None

    return {
        'WER (↓)':          round((sm(wers) or 1.0) * 100, 2),
        'CER (↓)':          round((sm(cers) or 1.0) * 100, 2),
        'MCD (↓)':          sm(mcds),
        'UTMOS (↑)':        sm(mos),
        'PESQ (↑)':         sm(pesqs),
        'STOI (↑)':         sm(stois),
        'F0-RMSE (↓)':      sm(f0s),
        'Speaking Rate':    sm(rates),
        'RTF (↓)':          sm(rtfs),
    }

print('Metric utilities ready.')

## 5. Model 1 — facebook/mms-tts-hin (Meta MMS-VITS)

The MMS-TTS Hindi model is part of Meta's Massively Multilingual Speech project.
It uses the VITS end-to-end TTS architecture — a conditional VAE with flow-based modules and
HiFi-GAN decoder — trained specifically on Hindi text, loaded entirely through HuggingFace
Transformers with no extra dependencies.

**Sample rate:** 16 kHz  |  **Params:** ~56M  |  **License:** CC-BY-NC 4.0


In [ ]:
from transformers import VitsModel, AutoTokenizer
from torch import set_grad_enabled

MMS_SR = 16000
print('Loading facebook/mms-tts-hin ...')
mms_tok   = AutoTokenizer.from_pretrained('facebook/mms-tts-hin')
mms_model = VitsModel.from_pretrained('facebook/mms-tts-hin').to(DEVICE)
mms_model.eval()
print('MMS-VITS loaded.')

mms_wavs, mms_times = [], []
mdir = OUT_DIR / 'mms_vits'; mdir.mkdir(exist_ok=True)

for i, text in enumerate(tqdm(texts, desc='MMS-VITS')):
    inputs = mms_tok(text, return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = mms_model(**inputs)
    mms_times.append(time.time() - t0)
    wav = out.waveform[0].cpu().numpy().astype(np.float32)
    mms_wavs.append(wav)
    sf.write(mdir / f'{i:04d}.wav', wav, MMS_SR)

avg_rtf = np.mean([t / (len(w)/MMS_SR) for t, w in zip(mms_times, mms_wavs)])
print(f'MMS-VITS done. Avg RTF = {avg_rtf:.3f}')
print(f'Sample duration: {np.mean([len(w)/MMS_SR for w in mms_wavs]):.2f}s')

In [ ]:
mms_metrics = evaluate_model('MMS-VITS', mms_wavs, MMS_SR, texts, ref_audios, mms_times)
print('MMS-VITS results:')
for k, v in mms_metrics.items():
    print(f'  {k}: {v}')

## 6. Model 2 — ai4bharat/indic-parler-tts (Indic Parler-TTS)

Indic Parler-TTS Mini is a HuggingFace x AI4Bharat collaboration.
It officially supports 20 Indic languages including Hindi, and is controlled via a natural
language description caption specifying speaker gender, speed, pitch, and clarity.

The model auto-detects the input language from the Devanagari script — no language code needed.

**Sample rate:** 44.1 kHz  |  **Params:** ~880M  |  **License:** Apache 2.0


In [ ]:
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer as _ATok

PARLER_SR = 44100
PARLER_DESC = (
    'A female speaker delivers clear and natural Hindi speech at a moderate pace, '
    'with a neutral tone and very clear audio quality.'
)

print('Loading ai4bharat/indic-parler-tts ...')
parler_model = ParlerTTSForConditionalGeneration.from_pretrained(
    'ai4bharat/indic-parler-tts'
).to(DEVICE)
parler_model.eval()
parler_tok   = _ATok.from_pretrained('ai4bharat/indic-parler-tts')
parler_desc_tok = _ATok.from_pretrained(parler_model.config.text_encoder._name_or_path)
print('Indic Parler-TTS loaded.')

parler_wavs, parler_times = [], []
pdir = OUT_DIR / 'parler'; pdir.mkdir(exist_ok=True)

for i, text in enumerate(tqdm(texts, desc='Parler-TTS')):
    desc_ids = parler_desc_tok(PARLER_DESC, return_tensors='pt').to(DEVICE)
    text_ids  = parler_tok(text, return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        gen = parler_model.generate(
            input_ids=desc_ids.input_ids,
            attention_mask=desc_ids.attention_mask,
            prompt_input_ids=text_ids.input_ids,
            prompt_attention_mask=text_ids.attention_mask,
        )
    parler_times.append(time.time() - t0)
    wav = gen.cpu().numpy().squeeze().astype(np.float32)
    parler_wavs.append(wav)
    sf.write(pdir / f'{i:04d}.wav', wav, PARLER_SR)

avg_rtf = np.mean([t / (len(w)/PARLER_SR) for t, w in zip(parler_times, parler_wavs)])
print(f'Parler-TTS done. Avg RTF = {avg_rtf:.3f}')

In [ ]:
parler_metrics = evaluate_model('Parler-TTS', parler_wavs, PARLER_SR, texts, ref_audios, parler_times)
print('Parler-TTS results:')
for k, v in parler_metrics.items():
    print(f'  {k}: {v}')

## 7. Model 3 — hexgrad/Kokoro-82M (Hindi voice)

Kokoro is an 82M-parameter model ranked #1 on TTS Arena. With `misaki[hi]` installed it
supports Hindi via `lang_code='h'` and a Hindi voice preset.
Apache 2.0, no gating, no custom toolkit.

**Sample rate:** 24 kHz  |  **Params:** 82M  |  **License:** Apache 2.0


In [ ]:
from kokoro import KPipeline

KOKORO_SR = 24000
print('Loading Kokoro (Hindi) ...')
# lang_code='h' activates Hindi phonemisation via misaki[hi]
kokoro_pipe = KPipeline(lang_code='h')
print('Kokoro loaded.')

kokoro_wavs, kokoro_times = [], []
kdir = OUT_DIR / 'kokoro'; kdir.mkdir(exist_ok=True)

for i, text in enumerate(tqdm(texts, desc='Kokoro-Hindi')):
    t0 = time.time()
    chunks = []
    # 'hf_alpha' is the default Hindi female voice in Kokoro
    for _, _, audio_chunk in kokoro_pipe(text, voice='hf_alpha', speed=1.0):
        if audio_chunk is not None:
            chunks.append(np.asarray(audio_chunk, dtype=np.float32))
    kokoro_times.append(time.time() - t0)
    wav = np.concatenate(chunks) if chunks else np.zeros(KOKORO_SR, dtype=np.float32)
    kokoro_wavs.append(wav)
    sf.write(kdir / f'{i:04d}.wav', wav, KOKORO_SR)

avg_rtf = np.mean([t / (len(w)/KOKORO_SR) for t, w in zip(kokoro_times, kokoro_wavs)])
print(f'Kokoro done. Avg RTF = {avg_rtf:.3f}')

In [ ]:
kokoro_metrics = evaluate_model('Kokoro', kokoro_wavs, KOKORO_SR, texts, ref_audios, kokoro_times)
print('Kokoro results:')
for k, v in kokoro_metrics.items():
    print(f'  {k}: {v}')

## 8. Aggregate Results

In [ ]:
all_results = {
    'MMS-VITS (Meta)':        mms_metrics,
    'Indic Parler-TTS (AI4B)': parler_metrics,
    'Kokoro-82M':              kokoro_metrics,
}

results_df = pd.DataFrame(all_results).T
results_df.index.name = 'Model'

higher_better = ['UTMOS (↑)', 'PESQ (↑)', 'STOI (↑)']
lower_better  = ['WER (↓)', 'CER (↓)', 'MCD (↓)', 'F0-RMSE (↓)', 'RTF (↓)']

print('=' * 72)
print('  HINDI TTS BENCHMARK RESULTS')
print('=' * 72)
print(results_df.to_string())
print()
print('  (↑) higher is better    (↓) lower is better')
print('  Speaking Rate: energy onsets/sec (informational only)')

## 9. Rankings

In [ ]:
rank_df = results_df.copy().astype(float)
for col in higher_better:
    if col in rank_df: rank_df[col] = rank_df[col].rank(ascending=False).astype(int)
for col in lower_better:
    if col in rank_df: rank_df[col] = rank_df[col].rank(ascending=True).astype(int)
if 'Speaking Rate' in rank_df:
    rank_df['Speaking Rate'] = (results_df['Speaking Rate'].astype(float) - 4.0).abs().rank().astype(int)
rank_df['Avg Rank'] = rank_df.mean(axis=1).round(2)
rank_df = rank_df.sort_values('Avg Rank')
print('=' * 72)
print('RANKINGS (1 = best per metric)')
print('=' * 72)
print(rank_df.to_string())
print(f'\nOverall winner: {rank_df["Avg Rank"].idxmin()}')

## 10. Visualisations

In [ ]:
models      = list(all_results.keys())
short       = ['MMS-VITS', 'Parler-TTS', 'Kokoro']
colors      = ['#4C72B0', '#DD8452', '#55A868']
best_color  = '#2ca02c'

plot_metrics = [
    ('WER (↓)',     'WER (%)',          False),
    ('CER (↓)',     'CER (%)',          False),
    ('MCD (↓)',     'MCD (dB)',         False),
    ('UTMOS (↑)',   'UTMOS (1-5)',      True),
    ('PESQ (↑)',    'PESQ score',       True),
    ('STOI (↑)',    'STOI (0-1)',       True),
    ('F0-RMSE (↓)', 'F0 RMSE (cents)', False),
    ('RTF (↓)',     'Real-Time Factor', False),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Hindi TTS Benchmark — All Metrics', fontsize=14, fontweight='bold', y=1.01)
axes = axes.flatten()

for ax, (metric, ylabel, hi_good) in zip(axes, plot_metrics):
    vals = [all_results[m].get(metric) for m in models]
    if all(v is not None for v in vals):
        best = vals.index(max(vals) if hi_good else min(vals))
    else:
        best = -1
    bar_cols = [best_color if j == best else c for j, c in enumerate(colors)]
    bars = ax.bar(range(len(models)), vals, color=bar_cols, edgecolor='white', width=0.55)
    for bar, val in zip(bars, vals):
        if val is not None:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + bar.get_height()*0.02,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    arrow = 'higher better' if hi_good else 'lower better'
    ax.set_title(f'{metric}\n({arrow})', fontsize=10, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(short, rotation=12, ha='right', fontsize=9)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('hindi_tts_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hindi_tts_all_metrics.png')

In [ ]:
radar_cols  = ['WER (↓)', 'CER (↓)', 'MCD (↓)', 'UTMOS (↑)', 'PESQ (↑)', 'STOI (↑)', 'RTF (↓)']
radar_df    = results_df[radar_cols].astype(float)
norm_df     = radar_df.copy()

for col in radar_cols:
    lo, hi = radar_df[col].min(), radar_df[col].max()
    if hi > lo:
        norm_df[col] = (radar_df[col] - lo)/(hi - lo) if col in higher_better \
                       else 1 - (radar_df[col] - lo)/(hi - lo)
    else:
        norm_df[col] = 0.5

labels  = ['WER\n(inv)', 'CER\n(inv)', 'MCD\n(inv)', 'UTMOS', 'PESQ', 'STOI', 'RTF\n(inv)']
n       = len(labels)
angles  = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for (model, row), color, sname in zip(norm_df.iterrows(), colors, short):
    vals = row.tolist() + row.tolist()[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=sname, color=color)
    ax.fill(angles, vals, alpha=0.08, color=color)

ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Hindi TTS — Normalised Radar\n(1.0 = best on each axis)',
             fontsize=12, fontweight='bold', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig('hindi_tts_radar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Waveform + Mel spectrogram for one example across all models
sample_idx = 0
all_wavs_plot = [
    ('Reference (IndicTTS)', ref_audios[sample_idx],   REF_SR,    'black'),
    ('MMS-VITS',            mms_wavs[sample_idx],      MMS_SR,    colors[0]),
    ('Parler-TTS',          parler_wavs[sample_idx],   PARLER_SR, colors[1]),
    ('Kokoro',              kokoro_wavs[sample_idx],   KOKORO_SR, colors[2]),
]

fig, axes = plt.subplots(len(all_wavs_plot), 2, figsize=(14, 3.5*len(all_wavs_plot)))
fig.suptitle(
    f'Waveform & Mel Spectrogram Comparison\nText: "{texts[sample_idx][:60]}"',
    fontsize=11, fontweight='bold'
)

for row, (name, wav, sr, c) in enumerate(all_wavs_plot):
    t = np.linspace(0, len(wav)/sr, len(wav))
    axes[row,0].plot(t, wav, linewidth=0.4, color=c)
    axes[row,0].set_title(f'{name}', fontsize=9, fontweight='bold')
    axes[row,0].set_xlabel('Time (s)'); axes[row,0].set_ylabel('Amplitude')
    axes[row,0].spines[['top','right']].set_visible(False)

    mel = librosa.power_to_db(
        librosa.feature.melspectrogram(y=wav, sr=sr, n_mels=80), ref=np.max)
    img = librosa.display.specshow(mel, sr=sr, x_axis='time', y_axis='mel',
                                   ax=axes[row,1], cmap='magma')
    axes[row,1].set_title(f'{name} — Mel Spec', fontsize=9, fontweight='bold')
    fig.colorbar(img, ax=axes[row,1], format='%+2.0f dB')

plt.tight_layout()
plt.savefig('hindi_waveforms_specs.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(all_wavs_plot), figsize=(18, 4), sharey=True)
fig.suptitle('F0 Pitch Contour — Sample 0', fontsize=12, fontweight='bold')

for ax, (name, wav, sr, c) in zip(axes, all_wavs_plot):
    f0, _, _ = librosa.pyin(wav, fmin=60, fmax=500, sr=sr, hop_length=256)
    times = librosa.times_like(f0, sr=sr, hop_length=256)
    ax.plot(times, f0, color=c, linewidth=1.5)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('F0 (Hz)')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('hindi_f0_contours.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# RTF vs WER trade-off scatter
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Speed vs Quality Trade-offs', fontsize=13, fontweight='bold')

for m, c, sn in zip(models, colors, short):
    rtf = all_results[m].get('RTF (↓)')
    wer = all_results[m].get('WER (↓)')
    cer = all_results[m].get('CER (↓)')
    if rtf and wer:
        axes[0].scatter(rtf, wer, color=c, s=200, zorder=5)
        axes[0].annotate(sn, (rtf, wer), textcoords='offset points', xytext=(8,4), fontsize=9)
    if rtf and cer:
        axes[1].scatter(rtf, cer, color=c, s=200, zorder=5, label=sn)
        axes[1].annotate(sn, (rtf, cer), textcoords='offset points', xytext=(8,4), fontsize=9)

for ax, ylabel in zip(axes, ['WER % (↓)', 'CER % (↓)']):
    ax.axvline(1.0, color='red', linestyle='--', alpha=0.4, label='RTF=1')
    ax.set_xlabel('RTF — Real-Time Factor (↓)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)

axes[0].set_title('RTF vs WER'); axes[1].set_title('RTF vs CER')
plt.tight_layout()
plt.savefig('hindi_rtf_vs_errors.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-sample CER distribution + gender breakdown
model_wav_triples = [
    ('MMS-VITS',   mms_wavs,    MMS_SR),
    ('Parler-TTS', parler_wavs, PARLER_SR),
    ('Kokoro',     kokoro_wavs, KOKORO_SR),
]

per_sample = {'text': texts, 'gender': genders}
for mname, wavs, sr in model_wav_triples:
    cers_ps = []
    for wav, text in zip(wavs, texts):
        s = librosa.resample(wav, orig_sr=sr, target_sr=REF_SR) if sr != REF_SR else wav
        hyp = _transcribe_hindi(s, REF_SR)
        try: cers_ps.append(min(_cer_fn(text.strip(), hyp), 1.0) * 100)
        except: cers_ps.append(100.0)
    per_sample[mname] = cers_ps

ps_df = pd.DataFrame(per_sample)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Per-Sample CER Analysis', fontsize=13, fontweight='bold')

# Distribution
for (mname, _, __), c in zip(model_wav_triples, colors):
    axes[0].hist(ps_df[mname], bins=20, alpha=0.55, label=mname, color=c, edgecolor='white')
axes[0].set_xlabel('CER (%)'); axes[0].set_ylabel('Count')
axes[0].set_title('CER Distribution'); axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Gender breakdown
gender_cer = ps_df.groupby('gender')[['MMS-VITS','Parler-TTS','Kokoro']].mean()
x = np.arange(len(gender_cer))
w = 0.25
for j, (mname, c) in enumerate(zip(['MMS-VITS','Parler-TTS','Kokoro'], colors)):
    axes[1].bar(x + j*w, gender_cer[mname], w, label=mname, color=c, edgecolor='white')
axes[1].set_xticks(x + w); axes[1].set_xticklabels(['Female','Male'])
axes[1].set_ylabel('Mean CER (%)'); axes[1].set_title('CER by Gender')
axes[1].legend(fontsize=8); axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('hindi_cer_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

ps_df['avg_cer'] = ps_df[['MMS-VITS','Parler-TTS','Kokoro']].mean(axis=1)
print('5 hardest sentences (highest avg CER):')
for _, row in ps_df.nlargest(5, 'avg_cer').iterrows():
    print(f'  [{row.gender}] "{row.text[:70]}"')
    for mname, _, __ in model_wav_triples:
        print(f'      {mname}: {row[mname]:.1f}%')

In [ ]:
# CER vs text length — do longer sentences degrade quality?
ps_df['char_len'] = ps_df['text'].str.len()
bins  = [0, 20, 40, 60, 80, 200]
lbls  = ['0-20', '21-40', '41-60', '61-80', '80+']
ps_df['len_bin'] = pd.cut(ps_df['char_len'], bins=bins, labels=lbls)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle('CER vs Text Length', fontsize=12, fontweight='bold')

for ax, (mname, c) in zip(axes, zip(['MMS-VITS','Parler-TTS','Kokoro'], colors)):
    grp = ps_df.groupby('len_bin', observed=True)[mname].mean()
    ax.bar(grp.index, grp.values, color=c, edgecolor='white', width=0.6)
    ax.set_title(mname, fontsize=10, fontweight='bold')
    ax.set_xlabel('Text Length (chars)'); ax.set_ylabel('Mean CER (%)')
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('hindi_cer_vs_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Save Results

In [ ]:
results_df.to_csv('hindi_tts_benchmark_results.csv')
ps_df.to_csv('hindi_tts_per_sample.csv', index=False)
print('Saved: hindi_tts_benchmark_results.csv')
print('Saved: hindi_tts_per_sample.csv')
print()
print('=' * 72)
print('FINAL SUMMARY')
print('=' * 72)
summary = ['WER (↓)', 'CER (↓)', 'MCD (↓)', 'UTMOS (↑)', 'PESQ (↑)', 'STOI (↑)', 'RTF (↓)']
print(results_df[summary].to_string())

## 12. Discussion

### Metric Guide for Hindi TTS

| Metric | Why it matters for Hindi | Library |
|---|---|---|
| **CER** | Devanagari has compound characters — char-level errors better capture mispronunciation | `jiwer` + Whisper |
| **WER** | Coarser word-level view; good complement to CER | `jiwer` + Whisper |
| **MCD** | Spectral distance from reference — measures acoustic fidelity (speaker-dependent) | `librosa` + `dtw-python` |
| **UTMOS** | Reference-free neural MOS — most reliable single metric for naturalness | `utmos` |
| **PESQ** | ITU-T telephony quality standard — correlates with perceptual clarity | `pesq` |
| **STOI** | Intelligibility objective measure — captures clarity independent of speaker | `pystoi` |
| **F0 RMSE** | Pitch accuracy in cents — measures prosodic naturalness | `librosa` |
| **RTF** | Real-time factor — synthesis time / audio duration | wall clock |

### Caveats
- **MCD, PESQ, STOI** compare against the IndicTTS-Hindi reference speaker. All three benchmarked
  models produce a **different speaker** from the reference, so these scores reflect speaker
  distance as much as quality. **UTMOS is the fairest single-model quality indicator.**
- **Whisper large-v3** handles Hindi Devanagari well but may normalise some spellings
  differently from the IndicTTS transcriptions, slightly inflating WER/CER.
- **RTF is hardware-dependent.** Run all cells on the same Kaggle kernel for fair comparison.

### Expected ordering
- **MMS-VITS**: Fastest (RTF << 1), smallest model, decent intelligibility
- **Kokoro**: Best naturalness (UTMOS), very fast, Apache 2.0
- **Indic Parler-TTS**: Most expressive, controllable via caption, but slower

### Reproducibility
- `N_SAMPLES = 50` (set to 200+ for publication-quality results)
- Dataset seed: 42; stratified 50% female / 50% male
- All models are public, pip-installable, no HuggingFace tokens required
